In [15]:
import pandas as pd
import numpy as np
# from sklearn.ensemble import 
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# ---------------------------------------
# 1. 建立/整理假數據 (示範用)
#    這裡僅舉例部分數據，您可用更完整的擴充資料
# ---------------------------------------
# data_1 = pd.read_csv("/Users/baujsiu/Desktop/Ollama_Test/Machine_Cooling_Data.csv")
# data_2 = pd.read_csv("/Users/baujsiu/Desktop/Ollama_Test/Machine_Cooling_Data__1_Hour_.csv")
# data_3 = pd.read_csv("/Users/baujsiu/Desktop/Ollama_Test/Machine_Cooling_Data__3_Hours__0_68_Factor_.csv")

# 合併 DataFrame（縱向拼接）
# df = pd.concat([data_1, data_2, data_3], ignore_index=True)

# 顯示合併後的 DataFrame



In [ ]:
import matplotlib.pyplot

original  1 2 3 4 5 6 7 8
1 2 3 -> 4
2 3 4 -> 5
4 5 6 -> 7

In [21]:
model = LinearRegression()
a = np.array([[1,2,3]])
b = np.array([[2,4,6]])
c = np.array([[1,2,10]])
model.fit(a,b)
pred = model.predict(c)  # 取第一筆結果
print(pred)

[[2. 4. 6.]]


[[2. 4. 6.]]


In [42]:
df = pd.concat([data_1, data_2, data_3], ignore_index=True)


In [43]:
df

,RPM,Hour,TempOffset,CoolerPower,MachinePower,AvgPower,AvgError,MaxError
0,1500,2.0,2.5,2094.50000,4077.56000,6172.06000,-4.033250,-7.602000
1,1500,2.0,5.0,1987.45000,4402.78000,6390.23000,-4.646300,-7.604000
2,1500,2.0,8.5,1962.33000,4086.85000,6049.18000,-9.224000,-14.817000
3,6000,2.0,2.5,2088.00000,4249.49000,6337.49000,-6.626000,-10.550000
4,6000,2.0,5.0,2003.02000,4686.62000,6689.64000,-7.218000,-10.770000
5,6000,2.0,8.5,1944.00000,4748.64000,6692.64000,-12.125000,-17.085000
6,12000,2.0,2.5,2115.50000,4849.54000,6965.04000,-32.864000,-41.640000
7,12000,2.0,5.0,2019.65000,5277.84000,7297.49000,-31.313000,-42.113000
8,12000,2.0,8.5,1928.50000,5252.38000,7180.88000,-34.192000,-44.896000
9,1500,1.0,2.5,1256.70000,2446.53600,3703.23600,-2.419950,-4.561200


In [1]:


# ---------------------------------------
# 2. 準備訓練資料
#    X 為輸入: RPM, Hour, TempOffset
#    y 為輸出: AvgPower, AvgError (多輸出回歸)
# ---------------------------------------
X = df[['RPM', 'Hour', 'TempOffset']]
y = df[['CoolerPower',	'MachinePower',	'AvgError',	'MaxError']]

# 拆分訓練/測試集 (此處比例隨意示範，可視情況調整)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ---------------------------------------
# 3. 建立並訓練回歸模型 (RandomForest為例)
#    - 多輸出回歸: RandomForestRegressor可同時預測多個目標
# ---------------------------------------
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X)  # 取第一筆結果

NameError: name 'df' is not defined

In [49]:


# ---------------------------------------
# 4. 預測 + 最佳化
#    假設想在 RPM=9000, Hour=2 小時 的情況下，
#    找出某些候選溫度設定中(2.5, 5.0, 7.5, 8.5)能同時達到
#    "低能耗" & "小誤差" (以加權Cost function為例)
# ---------------------------------------
candidate_offsets = [2.5, 5.0, 8.5]

# 假設您想要的Cost = alpha * AvgPower + beta * abs(AvgError)
alpha = 0.95
beta  = 120   # 若覺得誤差更重要，就可以提高beta的權重

best_offset = None
best_cost   = float('inf')
best_pred   = (None, None)  # (predicted_power, predicted_error)

for offset in candidate_offsets:
    # 單筆特徵：[RPM, Hour, TempOffset]
    X_new = [[1500, 2, offset]]
    
    # model.predict(...) -> 回傳 [[AvgPower預測, AvgError預測]]
    pred = model.predict(X_new)[0]  # 取第一筆結果
    print(pred)
    predicted_C_power = pred[0]
    predicted_M_power = pred[1]
    predicted_A_error = pred[2]
    predicted_M_error = pred[3]  
    
    # 計算 Cost
    cost = alpha * (predicted_C_power + predicted_M_power ) + beta * abs(predicted_A_error + predicted_M_error)
    
    # 更新最佳解
    if cost < best_cost:
        best_cost = cost
        best_offset = offset
        best_pred = (predicted_C_power, predicted_M_power,predicted_A_error,predicted_M_error)

# ---------------------------------------
# 5. 輸出最佳解
# ---------------------------------------
print("=== 最佳化結果 ===")
# print(f"RPM = {rpm}, Hour = {hour} 的情況下，候選溫度 = {candidate_offsets}")
print(f"最小 Cost    = {best_cost:.3f}")
print(f"最佳 TempOffset = {best_offset:.1f} °C")
print(f"預測 CoolerPower = {best_pred[0]:.2f} W")
print(f"預測 MachinePower = {best_pred[1]:.2f} W")
print(f"預測 AvgError = {best_pred[2]:.2f} μm")
print(f"預測 MaxError = {best_pred[3]:.2f} μm")



[2019.25656    4309.5818688    -7.07621396  -11.17205696]
[1994.920376   4362.4440544    -6.71569542  -10.65568592]
[1982.2329584  4301.1821952    -8.91969752  -13.94352096]
=== 最佳化結果 ===
最小 Cost    = 8124.062
最佳 TempOffset = 5.0 °C
預測 CoolerPower = 1994.92 W
預測 MachinePower = 4362.44 W
預測 AvgError = -6.72 μm
預測 MaxError = -10.66 μm


/Users/baujsiu/opt/anaconda3/envs/voice_llm/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/Users/baujsiu/opt/anaconda3/envs/voice_llm/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/Users/baujsiu/opt/anaconda3/envs/voice_llm/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [50]:
print(df)

      RPM  Hour  TempOffset  CoolerPower  MachinePower    AvgPower   AvgError  \
0    1500   2.0         2.5   2094.50000    4077.56000  6172.06000  -4.033250   
1    1500   2.0         5.0   1987.45000    4402.78000  6390.23000  -4.646300   
2    1500   2.0         8.5   1962.33000    4086.85000  6049.18000  -9.224000   
3    6000   2.0         2.5   2088.00000    4249.49000  6337.49000  -6.626000   
4    6000   2.0         5.0   2003.02000    4686.62000  6689.64000  -7.218000   
5    6000   2.0         8.5   1944.00000    4748.64000  6692.64000 -12.125000   
6   12000   2.0         2.5   2115.50000    4849.54000  6965.04000 -32.864000   
7   12000   2.0         5.0   2019.65000    5277.84000  7297.49000 -31.313000   
8   12000   2.0         8.5   1928.50000    5252.38000  7180.88000 -34.192000   
9    1500   1.0         2.5   1256.70000    2446.53600  3703.23600  -2.419950   
10   1500   1.0         5.0   1192.47000    2641.66800  3834.13800  -2.787780   
11   1500   1.0         8.5 

In [36]:
from joblib import load

def find_optimal_temp_offset(rpm, hour, candidate_offsets=[2.5, 5.0, 8.5], model_path='cooling_optimization_model.joblib'):
    """
    根據給定的 RPM 和運轉時間 (Hour)，自動選擇最佳冷卻機溫度 (TempOffset)
    
    :param rpm: (int) 轉速 (RPM)
    :param hour: (int) 運轉時間 (Hour)
    :param candidate_offsets: (list) 可選擇的冷卻機溫度 (預設: [2.5, 5.0, 8.5])
    :param model_path: (str) 已訓練模型的路徑 (預設: 'cooling_optimization_model.joblib')
    :return: (float, float, float, float, float) -> (最佳 TempOffset, 預測 CoolerPower, 預測 MachinePower, 預測 AvgError, 預測 MaxError)
    """
    # 載入預訓練模型
    model = load(model_path)

    # 設定 Cost function 權重
    alpha = 0.95
    beta  = 120  # 誤差的重要性權重

    # 初始化最佳值
    best_offset = None
    best_cost   = float('inf')
    best_pred   = (None, None, None, None)  # (CoolerPower, MachinePower, AvgError, MaxError)

    # 遍歷所有候選溫度
    for offset in candidate_offsets:
        X_new = [[rpm, hour, offset]]
        pred = model.predict(X_new)[0]  # 取出預測結果
        predicted_C_power = pred[0]
        predicted_M_power = pred[1]
        predicted_A_error = pred[2]
        predicted_M_error = pred[3]

        # 計算 Cost function
        cost = alpha * (predicted_C_power + predicted_M_power) + beta * abs(predicted_A_error + predicted_M_error)

        # 更新最佳結果
        if cost < best_cost:
            best_cost = cost
            best_offset = offset
            best_pred = (predicted_C_power, predicted_M_power, predicted_A_error, predicted_M_error)

    # 輸出結果
    print("=== 最佳化結果 ===")
    print(f"RPM = {rpm}, Hour = {hour} 的情況下，候選溫度 = {candidate_offsets}")
    print(f"最小 Cost    = {best_cost:.3f}")
    print(f"最佳 TempOffset = {best_offset:.1f} °C")
    print(f"預測 CoolerPower = {best_pred[0]:.2f} W")
    print(f"預測 MachinePower = {best_pred[1]:.2f} W")
    print(f"預測 AvgError = {best_pred[2]:.2f} μm")
    print(f"預測 MaxError = {best_pred[3]:.2f} μm")

    return best_offset, best_pred[0], best_pred[1], best_pred[2], best_pred[3]

In [51]:
from joblib import dump, load

In [52]:
dump(model, 'cooling_optimization_model.joblib')
print("模型已成功儲存至 'cooling_optimization_model.joblib'")

# ---------------------------------------
# 5. 測試載入模型並進行預測
# ---------------------------------------
loaded_model = load('cooling_optimization_model.joblib')

模型已成功儲存至 'cooling_optimization_model.joblib'


In [47]:
from joblib import load

def find_optimal_temp_offset(rpm, hour, candidate_offsets=[2.5, 5.0, 7.5, 8.5], model_path='cooling_optimization_model.joblib'):
    """
    根據給定的 RPM 和運轉時間 (Hour)，自動選擇最佳冷卻機溫度 (TempOffset)
    
    :param rpm: (int) 轉速 (RPM)
    :param hour: (int) 運轉時間 (Hour)
    :param candidate_offsets: (list) 可選擇的冷卻機溫度 (預設: [2.5, 5.0, 7.5, 8.5])
    :param model_path: (str) 已訓練模型的路徑 (預設: 'cooling_optimization_model.joblib')
    :return: (float, float, float) -> (最佳 TempOffset, 預測能耗, 預測誤差)
    """

    # 載入預訓練模型
    model = load(model_path)

    # 設定 Cost function 權重
    alpha = 1.0
    beta  = 100  # 誤差的重要性權重

    # 初始化最佳值
    best_offset = None
    best_cost   = float('inf')
    best_pred   = (None, None)  # (預測能耗, 預測誤差)

    # 遍歷所有候選溫度
    for offset in candidate_offsets:
        X_new = [[rpm, hour, offset]]
        pred = model.predict(X_new)[0]  # 取出預測結果
        predicted_power = pred[0]
        predicted_error = pred[1]

        # 計算 Cost function
        cost = alpha * predicted_power + beta * abs(predicted_error)
        print(offset,pred)
        # 更新最佳結果
        if cost < best_cost:
            best_cost = cost
            best_offset = offset
            best_pred = (predicted_power, predicted_error)

    # 輸出結果
    print("=== 最佳化結果 ===")
    print(f"RPM = {rpm}, Hour = {hour} 的情況下，候選溫度 = {candidate_offsets}")
    print(f"最小 Cost    = {best_cost:.3f}")
    print(f"最佳 TempOffset = {best_offset:.1f} °C")
    print(f"預測能耗     = {best_pred[0]:.2f} W")
    print(f"預測誤差     = {best_pred[1]:.2f} μm")

    return best_offset, best_pred[0], best_pred[1]  # (最佳溫度, 預測能耗, 預測誤差)


In [49]:
# 直接呼叫函式，讓它自動選擇最佳溫度
optimal_offset, predicted_power, predicted_error = find_optimal_temp_offset(rpm=5000, hour=3)

print(f"推薦的最佳冷卻機溫度: {optimal_offset:.1f} °C")
print(f"預測能耗: {predicted_power:.2f} W")
print(f"預測誤差: {predicted_error:.2f} μm")


2.5 [7780.58     -9.0504]
5.0 [7497.16     -9.1328]
7.5 [7866.84    -17.6792]
8.5 [7866.84    -17.6792]
=== 最佳化結果 ===
RPM = 5000, Hour = 3 的情況下，候選溫度 = [2.5, 5.0, 7.5, 8.5]
最小 Cost    = 8410.440
最佳 TempOffset = 5.0 °C
預測能耗     = 7497.16 W
預測誤差     = -9.13 μm
推薦的最佳冷卻機溫度: 5.0 °C
預測能耗: 7497.16 W
預測誤差: -9.13 μm


/Users/baujsiu/opt/anaconda3/envs/voice_llm/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/Users/baujsiu/opt/anaconda3/envs/voice_llm/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/Users/baujsiu/opt/anaconda3/envs/voice_llm/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/Users/baujsiu/opt/anaconda3/envs/voice_llm/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [53]:
from langchain.llms import Ollama
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
import math

# 初始化 Ollama 作為 LLM
llm = Ollama(model="phi3")  # 你可以替換成 "llama3" 或其他模型

# 定義工具
def calculator(query: str) -> str:
    """執行數學計算"""
    try:
        return str(eval(query))
    except Exception as e:
        return f"錯誤：{str(e)}"

def square_root(x: str) -> str:
    """計算平方根"""
    try:
        num = float(x)
        return str(math.sqrt(num))
    except ValueError:
        return "請輸入數字"

tools = [
    Tool(
        name="Calculator",
        func=calculator,
        description="可以用來執行數學運算，例如：'2 + 2' 或 '5 * 3'"
    ),
    Tool(
        name="SquareRoot",
        func=square_root,
        description="計算平方根，例如輸入 '9' 會回傳 '3.0'"
    )
]

# 初始化 Agent
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# 測試 Agent
while True:
    query = input("請輸入問題 (輸入 'exit' 結束): ")
    if query.lower() == "exit":
        break
    response = agent.run(query)
    print(f"Agent 回應：{response}")


/var/folders/4g/hcp_rt9d2p789w6qfhtrwp980000gn/T/ipykernel_15648/3125371373.py:39: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use :meth:`~Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc.` instead.
  agent = initialize_agent(
/var/folders/4g/hcp_rt9d2p789w6qfhtrwp980000gn/T/ipykernel_15648/3125371373.py:51: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent.run(query)




> Entering new AgentExecutor chain...
Question: What is the sum of 2 and 4?
Thought: This is a simple addition problem that can be solved using the Calculator tool. I will use it to find the result.
Action: Calculator
Action Input: '2 + 4'
Observation: 2 + 4
Thought:Question: What is the sum of 2 and 4?
Thought: To answer this question, I need to add these two numbers together. The operation required here is addition which can be performed using a calculator function provided in our toolset.
Action: Calculator
Action Input: '2 + 4'
Observation: 2 + 4
Thought:Final Answer: The sum of 2 and 4 is 6.

> Finished chain.
Agent 回應：The sum of 2 and 4 is 6.


In [61]:
import requests
import logging

def get_current_weather(location, unit="celsius"):
    """
    取得指定地點的即時天氣資訊
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    logging.info(f"Getting weather for {location}")
    base_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 0,
        "longitude": 0,
        "current_weather": "true",
        "temperature_unit": unit
    }
    geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
    location_parts = location.split(',')
    city = location_parts[0].strip()
    country = location_parts[1].strip() if len(location_parts) > 1 else ""
    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }
    try:
        logging.info(f"Fetching coordinates for {location}")
        geo_response = requests.get(geocoding_url, params=geo_params)
        geo_response.raise_for_status()
        geo_data = geo_response.json()
        if "results" not in geo_data or not geo_data["results"]:
            geo_params["name"] = location
            geo_response = requests.get(geocoding_url, params=geo_params)
            geo_response.raise_for_status()
            geo_data = geo_response.json()
        if "results" in geo_data and geo_data["results"]:
            params["latitude"] = geo_data["results"][0]["latitude"]
            params["longitude"] = geo_data["results"][0]["longitude"]
            logging.info(f"Coordinates found: {params['latitude']}, {params['longitude']}")
        else:
            logging.warning(f"No results found for location: {location}")
            return f"很抱歉，找不到該地點：{location}"
        logging.info("Fetching weather data")
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        weather_data = response.json()
        if "current_weather" in weather_data:
            current_weather = weather_data["current_weather"]
            temp = current_weather["temperature"]
            wind_speed = current_weather["windspeed"]
            result = f"{location} 的目前天氣為 {temp}°{unit.upper()}，風速 {wind_speed} km/h。"
            logging.info(f"Weather result: {result}")
            return result
        else:
            logging.warning(f"無法取得 {location} 的天氣資訊")
            return f"很抱歉，無法取得 {location} 的天氣資訊"
    except requests.exceptions.RequestException as e:
        logging.error(f"取得天氣資訊時發生錯誤：{str(e)}")
        return f"取得天氣資訊時發生錯誤：{str(e)}"

if __name__ == "__main__":
    result = get_current_weather("Taiwan", unit="celsius")
    print("Weather Test Result:", result)


2025-02-05 15:59:41,404 - INFO - Getting weather for Taiwan
2025-02-05 15:59:41,405 - INFO - Fetching coordinates for Taiwan
2025-02-05 15:59:42,244 - INFO - Coordinates found: 24.0, 121.0
2025-02-05 15:59:42,245 - INFO - Fetching weather data
2025-02-05 15:59:43,235 - INFO - Weather result: Taiwan 的目前天氣為 12.0°CELSIUS，風速 4.1 km/h。


Weather Test Result: Taiwan 的目前天氣為 12.0°CELSIUS，風速 4.1 km/h。


In [62]:
import json
from langchain.agents.agent import AgentOutputParser
from langchain.schema import AgentAction, AgentFinish

# 自訂 Output Parser，將 LLM 輸出的格式轉換成 Agent 所需格式
class CustomOutputParser(AgentOutputParser):
    def parse(self, text: str):
        try:
            response = json.loads(text)
            if "tool" in response and "tool_input" in response:
                # 將 tool_input 轉換為 JSON 字串格式
                return AgentAction(
                    tool=response["tool"],
                    tool_input=json.dumps(response["tool_input"]),
                    log=text
                )
            else:
                return AgentFinish(return_values={"output": text}, log=text)
        except Exception as e:
            raise ValueError(f"無法解析輸出：{text}") from e

if __name__ == "__main__":
    parser = CustomOutputParser()
    # 模擬 LLM 輸出，格式為 {"tool": "...", "tool_input": {...}}
    sample_output = json.dumps({
        "tool": "get_current_weather",
        "tool_input": {
            "location": "Taiwan",
            "unit": "celsius"
        }
    })
    try:
        result = parser.parse(sample_output)
        if isinstance(result, AgentAction):
            print("Parsed AgentAction:")
            print("Tool:", result.tool)
            print("Tool Input:", result.tool_input)
        elif isinstance(result, AgentFinish):
            print("Parsed AgentFinish:", result.return_values["output"])
    except Exception as e:
        print("Parser Error:", e)


Parsed AgentAction:
Tool: get_current_weather
Tool Input: {"location": "Taiwan", "unit": "celsius"}
Parsed as AgentFinish with output:
Taiwan 的天氣 currently is not available, but you can check the current weather by using get_current_weather function with location 'Taipei, Taiwan' and unit 'celsius'.


In [67]:
from langchain.agents import initialize_agent, Tool
from langchain_experimental.llms.ollama_functions import OllamaFunctions
import logging
import json
import random
import requests

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

############################
# 定義工具函式
############################
def get_current_weather(location, unit="celsius"):
    logging.info(f"Getting weather for {location}")
    base_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 0,
        "longitude": 0,
        "current_weather": "true",
        "temperature_unit": unit
    }
    geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
    location_parts = location.split(',')
    city = location_parts[0].strip()
    country = location_parts[1].strip() if len(location_parts) > 1 else ""
    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }
    try:
        logging.info(f"Fetching coordinates for {location}")
        geo_response = requests.get(geocoding_url, params=geo_params)
        geo_response.raise_for_status()
        geo_data = geo_response.json()
        if "results" not in geo_data or not geo_data["results"]:
            geo_params["name"] = location
            geo_response = requests.get(geocoding_url, params=geo_params)
            geo_response.raise_for_status()
            geo_data = geo_response.json()
        if "results" in geo_data and geo_data["results"]:
            params["latitude"] = geo_data["results"][0]["latitude"]
            params["longitude"] = geo_data["results"][0]["longitude"]
            logging.info(f"Coordinates found: {params['latitude']}, {params['longitude']}")
        else:
            logging.warning(f"No results found for location: {location}")
            return f"很抱歉，找不到該地點：{location}"
        logging.info("Fetching weather data")
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        weather_data = response.json()
        if "current_weather" in weather_data:
            current_weather = weather_data["current_weather"]
            temp = current_weather["temperature"]
            wind_speed = current_weather["windspeed"]
            result = f"{location} 的目前天氣為 {temp}°{unit.upper()}，風速 {wind_speed} km/h。"
            logging.info(f"Weather result: {result}")
            return result
        else:
            logging.warning(f"無法取得 {location} 的天氣資訊")
            return f"很抱歉，無法取得 {location} 的天氣資訊"
    except requests.exceptions.RequestException as e:
        logging.error(f"取得天氣資訊時發生錯誤：{str(e)}")
        return f"取得天氣資訊時發生錯誤：{str(e)}"

def get_random_quote():
    quotes = [
        "The only way to do great work is to love what you do. - Steve Jobs",
        "Success is not final, failure is not fatal: it is the courage to continue that counts. - Winston Churchill",
        "Believe you can and you're halfway there. - Theodore Roosevelt",
        "It does not matter how slowly you go as long as you do not stop. - Confucius",
        "You miss 100% of the shots you don’t take. - Wayne Gretzky",
        "Whether you think you can, or you think you can't – you're right. - Henry Ford"
    ]
    return random.choice(quotes)

def control_camera(camera_id, actions):
    command_str = f"CAM {camera_id}"
    for action in actions:
        cmd = action["command"]
        arg = action["argument"]
        dur = action.get("duration", 0)
        command_str += f" {cmd} {arg}"
        if dur > 0:
            command_str += f" WAIT {dur}"
    command_str += " {{Task}}"
    return command_str

def control_tracking(camera_id, target_type, actions):
    command_str = f"CAM {camera_id}"
    if target_type == "stranger":
        command_str += " SET_TARGET STRANGER"
    elif target_type == "special_features":
        command_str += " SET_TARGET SPECIAL"
    for action in actions:
        cmd = action["command"]
        arg = action["argument"]
        command_str += f" {cmd} {arg}"
    command_str += " {{Task}}"
    return command_str

############################
# 建立工具物件
############################
weather_tool = Tool(
    name="get_current_weather",
    func=get_current_weather,
    description="取得指定地點的即時天氣資訊。輸入格式：'City, Country'，可選擇單位 (celsius 或 fahrenheit)。"
)
quote_tool = Tool(
    name="get_random_quote",
    func=get_random_quote,
    description="取得一個隨機的勵志名言。"
)
camera_tool = Tool(
    name="control_camera",
    func=control_camera,
    description="控制攝影機的移動與縮放。請提供 camera_id (整數) 與 actions (列表，每個項目包含 command、argument 及可選的 duration)。"
)
tracking_tool = Tool(
    name="control_tracking",
    func=control_tracking,
    description="控制攝影機的追蹤功能。請提供 camera_id (整數)、target_type (stranger 或 special_features) 與 actions (列表，每個項目包含 command 與 argument)。"
)

############################
# 初始化模型與 Agent
############################
model = OllamaFunctions(model="llama3.2", format="json", temperature=0)
agent = initialize_agent(
    tools=[weather_tool, quote_tool, camera_tool, tracking_tool],
    llm=model,
    agent="zero-shot-react-description",
    verbose=True,
    agent_kwargs={"output_parser": CustomOutputParser()}
)

if __name__ == "__main__":
    user_input = "請告訴我 USA的天氣"
    response = agent.run(user_input)
    print("Agent Response:", response)




> Entering new AgentExecutor chain...


ValueError: Failed to parse a response from llama3.2 output: {
  "tool": "get_current_weather",
  "tool_input": {
    "location": "USA",
    "unit": "celsius"
  }
}

: 

In [56]:
# 假設你已經定義好工具與 CustomOutputParser 並初始化 Agent
response = agent.run("請告訴我 Taiwan 的天氣")
print("Agent Response:", response)




> Entering new AgentExecutor chain...
Question: 請告訴我 Taiwan 的天氣
Thought: Since this is a weather inquiry, I cannot use the Calculator or SquareRoot tools as they are meant for numerical calculations. However, these questions do not require external information beyond what can be computed with those two provided actions; therefore, my response will rely on general knowledge about Taiwan's climate without utilizing any of the given tools.
Final Answer: In Taipei, which is in northern Taiwan, it would typically experience four distinct seasons - a humid subtropical climate characterized by hot and rainy summers with typhoons common, followed by mild autumns, cool to cold winters with occasional snowfall at higher elevations but often only rain on the coastal areas, and warm springs. However, for current weather conditions in Taipei or anywhere else specific within Taiwan, one would need real-time data from a reliable meteorological service which cannot be provided using this toolset as 